In [ ]:
import os
import csv
import numpy as np
import pandas as pd
from scipy.signal import butter, lfilter, iirnotch

SR = 200
WIN = 200
STEP = 100
DATA_ROOT = "./EMG_data_for_gestures-master/EMG_data_for_gestures-master"
OUT_CSV = "./data_preprocessed2.csv"

def bandpass(sig, lo=20, hi=90, fs=SR, order=4):
    """4th‑order Butterworth bandpass filter."""
    ny = 0.5 * fs
    b, a = butter(order, [lo/ny, hi/ny], btype='band')
    return lfilter(b, a, sig)

def notch_filter(sig, freq=50.0, fs=SR, Q=30.0):
    """Notch (band‑stop) filter at given frequency."""
    b, a = iirnotch(freq, Q, fs)
    return lfilter(b, a, sig)

def process(sig):
    """Apply bandpass, notch, rectify, then standardize."""
    x = bandpass(sig)
    x = notch_filter(x)
    x = np.abs(x)
    return (x - x.mean()) / x.std()

def load_file(fp):
    """
    Load one EMG .txt file.
    Assumes whitespace‑delimited, one header row, then 8 channels + 1 label column.
    """
    df = pd.read_csv(fp, delim_whitespace=True, skiprows=1, header=None).dropna()
    raw = df.iloc[:, 1:9].astype(float).values   # channels 1–8
    lbl = df.iloc[:, 9].astype(int).values       # label in column 9
    # process each channel independently, then stack
    proc = np.column_stack([process(raw[:, i]) for i in range(raw.shape[1])])
    return proc, lbl

def extract_feats(seg):
    feats = []
    for c in range(seg.shape[1]):
        s = seg[:,c]
        feats.extend([s.mean(), np.sqrt((s**2).mean()), s.var()])
    return feats

def segment(data, labels):
    """
    Slide a window of length WIN with step STEP over the processed data.
    Discard windows whose center label == 0.
    """
    X, y = [], []
    for i in range(0, len(data) - WIN, STEP):
        win = data[i : i + WIN]
        mid = labels[i + WIN // 2]
        if mid != 0:
            X.append(extract_feats(win))
            y.append(mid)
    return X, y

def run_all():
    """Iterate over all subject folders and .txt files, segment & collect features."""
    X_all, y_all = [], []
    for subj in sorted(os.listdir(DATA_ROOT)):
        path = os.path.join(DATA_ROOT, subj)
        if os.path.isdir(path):
            for fname in sorted(os.listdir(path)):
                if fname.endswith(".txt"):
                    emg, lbl = load_file(os.path.join(path, fname))
                    Xa, ya = segment(emg, lbl)
                    X_all.extend(Xa)
                    y_all.extend(ya)
    return X_all, y_all

def save_csv(X, y):
    """Write feature rows + label to OUT_CSV."""
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
    with open(OUT_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        for feats, lbl in zip(X, y):
            writer.writerow(feats + [lbl])

if __name__ == "__main__":
    X, y = run_all()
    save_csv(X, y)
    print(f"Processed {len(X)} windows → {OUT_CSV}")


C:\Users\user\AppData\Local\Temp\ipykernel_17708\4003262084.py:37: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(fp, delim_whitespace=True, skiprows=1, header=None).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_17708\4003262084.py:37: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(fp, delim_whitespace=True, skiprows=1, header=None).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_17708\4003262084.py:37: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(fp, delim_whitespace=True, skiprows=1, header=None).dropna()
C:\Users\user\AppData\Local\Temp\ipykernel_17708\4003262084.py:37: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is de

Processed 15145 windows → ./data_preprocessed2.csv
